In [1]:
import pandas as pd


In [2]:
import pandas as pd
import numpy as np
sales= pd.read_csv("Final_Feature_code.csv")
sales.drop('order_date',axis=1, inplace= True)
sales.drop('customer_id',axis=1, inplace= True)
sales.drop('Unnamed: 0',axis=1, inplace= True)
sales.columns


Index(['purchase_count', 'avg_quantity', 'max_quantity', 'p25_quantity',
       'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
       'total_spend', 'avg_spend',
       ...
       'max_pp_30d_mean', 'max_pp_30d_min', 'max_pp_30d_max', 'max_pp_30d_std',
       'max_pp_30d_var', 'max_pp_30d_p25', 'max_pp_30d_p50', 'max_pp_30d_p75',
       'max_pp_30d_p90', 'max_pp_30d_p95'],
      dtype='str', length=1030)

In [3]:
X_train= sales.drop('product_id',axis=1)
y_train = sales['product_id']

In [4]:
import lightgbm as lgb
import shap
import numpy as np
import pandas as pd

# ============================================================
# 1. TRAIN LIGHTGBM
# ============================================================

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
    n_jobs=4
)

model.fit(X_train, y_train)


# ============================================================
# 2. CALCULATE SHAP VALUES
# ============================================================

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X_train)


# ============================================================
# 3. HANDLE DIFFERENT SHAP OUTPUT FORMATS
# ============================================================

# Older SHAP versions:
# Binary classification can return [class_0, class_1]
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Convert to numpy array
shap_values = np.asarray(shap_values)

print("SHAP shape:", shap_values.shape)
print("X_train shape:", X_train.shape)


# ============================================================
# 4. GET 1 SHAP IMPORTANCE VALUE PER FEATURE
# ============================================================

if shap_values.ndim == 2:

    # Normal case:
    # (rows, features)

    importance = np.abs(shap_values).mean(axis=0)

elif shap_values.ndim == 3:

    # Possible format:
    # (rows, features, classes)

    importance = np.abs(shap_values).mean(axis=(0, 2))

else:

    raise ValueError(
        f"Unexpected SHAP shape: {shap_values.shape}"
    )


# Make sure importance is 1-dimensional
importance = np.asarray(importance).reshape(-1)


# ============================================================
# 5. CHECK FEATURE / SHAP LENGTH
# ============================================================

print("Number of features:", len(X_train.columns))
print("Number of SHAP importances:", len(importance))

if len(X_train.columns) != len(importance):
    raise ValueError(
        f"Mismatch: X_train has {len(X_train.columns)} features "
        f"but SHAP has {len(importance)} importance values."
    )


# ============================================================
# 6. CREATE SHAP IMPORTANCE DATAFRAME
# ============================================================

shap_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importance
})


# ============================================================
# 7. SORT BY SHAP IMPORTANCE
# ============================================================

shap_df = shap_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)


# ============================================================
# 8. CALCULATE IMPORTANCE %
# ============================================================

total_importance = shap_df["importance"].sum()

shap_df["importance_pct"] = (
    shap_df["importance"] / total_importance
) * 100


# ============================================================
# 9. CUMULATIVE IMPORTANCE
# ============================================================

shap_df["cumulative_pct"] = (
    shap_df["importance_pct"].cumsum()
)


# ============================================================
# 10. SELECT FEATURES COVERING 95%
# ============================================================

# Find first feature that reaches/exceeds 95%
cutoff_index = (
    shap_df["cumulative_pct"] >= 95
).idxmax()

top_95 = shap_df.iloc[
    :cutoff_index + 1
].copy()


# ============================================================
# 11. FINAL FEATURE LIST
# ============================================================

selected_features = top_95["feature"].tolist()


print("\n" + "=" * 60)
print("SHAP FEATURE SELECTION")
print("=" * 60)

print("Total features:", len(shap_df))
print("Selected features:", len(selected_features))
print(
    "Final cumulative importance:",
    round(top_95["cumulative_pct"].iloc[-1], 2),
    "%"
)

print("\nSelected Features:")
print(selected_features)


# ============================================================
# 12. SHOW TOP FEATURES
# ============================================================

print("\nTop 95% SHAP Features:")
print(
    top_95[
        [
            "feature",
            "importance",
            "importance_pct",
            "cumulative_pct"
        ]
    ].to_string(index=False)
)

C:\Users\Lakshmi Priya\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1657: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  _LGBMCheckClassificationTargets(y)


SHAP shape: (100, 1029, 66)
X_train shape: (100, 1029)
Number of features: 1029
Number of SHAP importances: 1029

SHAP FEATURE SELECTION
Total features: 1029
Selected features: 10
Final cumulative importance: 96.08 %

Selected Features:
['total_pad', 'total_pd', 'total_pp', 'total_spend', 'avg_quantity', 'avg_pad', 'avg_pd', 'avg_spend', 'avg_pp', 'max_quantity']

Top 95% SHAP Features:
     feature  importance  importance_pct  cumulative_pct
   total_pad    0.703569       21.353286       21.353286
    total_pd    0.651074       19.760067       41.113353
    total_pp    0.459854       13.956566       55.069918
 total_spend    0.431761       13.103923       68.173841
avg_quantity    0.381144       11.567701       79.741542
     avg_pad    0.133869        4.062916       83.804458
      avg_pd    0.130351        3.956147       87.760605
   avg_spend    0.095568        2.900494       90.661100
      avg_pp    0.093957        2.851581       93.512681
max_quantity    0.084703        2.570742